# 07주차: PyTorch CIFAR-10 모델 구조와 데이터 증강

## 학습 목표
CIFAR-10 컬러 이미지에 6주차와 동일한 `SmallCNN` 구조를 입력 채널만 3으로 바꾸어 그대로 적용해 보고, 더 깊은 `ImprovedCNN` 구조와 데이터 증강이 검증·테스트 성능에 어떤 영향을 주는지 비교합니다. 같은 데이터 분할과 같은 학습 설정(손실 함수, 옵티마이저, **에포크 수**)을 유지한 채 모델 구조와 데이터 전처리만 바꾸어, 성능 변화의 원인을 하나씩 구분해서 관찰하는 실험 설계를 연습합니다.

Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 세 실험을 모두 합쳐 T4 기준 15분 내외가 걸립니다. GPU가 없어도 CPU에서 실행되지만 훨씬 오래 걸리므로 반드시 T4를 켜세요.

데이터는 자동으로 내려받으므로 Google Drive 연결, Drive 마운트, 파일 업로드는 필요하지 않습니다. 다만 6주차와 달리 **Hugging Face Hub**에서 받습니다. 6주차 Fashion-MNIST는 30MB라 torchvision 기본 서버로도 10초면 끝났지만, CIFAR-10은 170MB이고 기본 서버(cs.toronto.edu)가 국내·Colab에서 매우 느려 다운로드에만 수십 분이 걸리는 경우가 있습니다. Hugging Face는 같은 데이터를 CDN으로 서빙해 10초 안팎이면 받아집니다. 받은 뒤에는 이미지를 한 번에 메모리로 펼쳐 두므로 학습 속도는 torchvision과 동일합니다.

## 관찰 질문
- 6주차의 `SmallCNN` 구조를 그대로 쓰는데 입력 채널만 3으로 바뀌면 어떤 층의 파라미터 수가 달라질까요?
- Fashion-MNIST의 흑백 의류 이미지와 달리, CIFAR-10의 컬러 사물 이미지는 왜 더 어려운 분류 문제일까요?
- 데이터 증강은 학습 정확도와 검증 정확도의 차이(과적합 정도)를 어떻게 바꿀까요?
- 데이터 증강의 효과는 왜 학습 초반이 아니라 **학습이 충분히 진행된 뒤에야** 드러날까요?


In [ ]:
# Colab에는 datasets가 기본 설치되어 있지 않을 수 있습니다. 이미 있으면 그냥 넘어갑니다.
# (로컬에서 uv로 환경을 만들었다면 이 셀은 실행하지 않아도 됩니다.)
try:
    import datasets
except ImportError:
    %pip install -q datasets


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from datasets import load_dataset

# ---------------------------------------------------------------------------
# 이 노트북의 텐서 차원 표기
#   B = 배치 크기(여기서는 128, 마지막 배치는 더 작을 수 있음)
#   CIFAR-10은 컬러라 채널이 3개, 크기는 32 x 32다.
#   이미지 텐서는 (B, 3, 32, 32), 라벨 텐서는 (B,), 모델 출력은 (B, 10).
#   6주차 Fashion-MNIST는 (B, 1, 28, 28)이었다. 채널과 해상도가 모두 커졌다.
# ---------------------------------------------------------------------------

def seed_everything(seed=42):
    """파이썬·numpy·PyTorch(CPU/GPU)의 난수 생성기를 모두 같은 시드로 고정한다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# ---------------------------------------------------------------------------
# 데이터 내려받기: 6주차의 torchvision 대신 Hugging Face Hub를 쓴다
#
# 6주차 Fashion-MNIST(30MB)는 torchvision 기본 서버가 충분히 빨랐지만,
# CIFAR-10(170MB)의 기본 서버(cs.toronto.edu)는 국내·Colab에서 매우 느려
# 다운로드에만 수십 분이 걸리는 경우가 있다.
# Hugging Face는 같은 데이터를 CDN으로 서빙하므로 10초 안팎이면 받아진다.
# ---------------------------------------------------------------------------

def choose_num_workers():
    """실행 환경의 CPU 개수에 맞춰 DataLoader 워커 수를 정한다.

    워커는 배치를 미리 준비해 두는 별도 프로세스다. Colab 런타임마다
    할당되는 vCPU 수가 다르므로(2개인 경우도, 8개인 경우도 있다) 고정값 대신
    실행할 때 세어서 정한다.

    - os.sched_getaffinity(0)은 "이 프로세스가 실제로 쓸 수 있는" 코어를 센다.
      Colab처럼 컨테이너로 코어를 제한하는 환경에서 os.cpu_count()는 호스트
      전체 코어를 세어 과대평가할 수 있다. 리눅스에만 있으므로 없으면 대체한다.
    - 워커를 1개만 쓰면 0개(메인 프로세스가 직접 준비)보다 오히려 느리다.
      병렬성은 없으면서 프로세스 간에 데이터를 넘기는 비용만 붙기 때문이다.
      그래서 코어가 2개 미만이면 아예 0으로 둔다.
    - 8을 넘겨도 이득이 거의 없다. 그 지점이면 이미 GPU 연산이 병목이다.
      코어 수보다 많은 워커를 만들면 오히려 느려지고 PyTorch가 경고도 낸다.
    """
    try:
        cores = len(os.sched_getaffinity(0))     # 리눅스(Colab 포함)
    except AttributeError:
        cores = os.cpu_count() or 1              # macOS, Windows 등
    return 0 if cores < 2 else min(8, cores)


def load_hf_images(repo_id, split):
    """HF Hub에서 데이터셋을 받아 (이미지 배열, 라벨 배열)로 메모리에 펼친다.

    반환값
      images: (N, 32, 32, 3) uint8 numpy 배열   labels: (N,) int64 numpy 배열

    HF는 이미지를 PNG로 압축해 저장하기 때문에, 학습 중 매번 꺼내 쓰면
    접근할 때마다 PNG를 푸느라 데이터 로딩이 2배 가까이 느려진다.
    그래서 여기서 한 번만 전부 풀어 numpy 배열로 만들어 둔다.
    CIFAR-10 학습셋은 이렇게 해도 146MB라 메모리에 충분히 올라간다.
    (torchvision의 datasets.CIFAR10도 내부적으로 똑같이 동작한다.)
    """
    rows = load_dataset(repo_id, split=split)
    # 'label'이 아닌 나머지 컬럼이 이미지 컬럼이다(CIFAR-10은 'img').
    image_key = [name for name in rows.column_names if name != "label"][0]
    images = np.stack([np.asarray(image) for image in rows[image_key]])   # (N, 32, 32, 3)
    labels = np.asarray(rows["label"])                                    # (N,)
    return images, labels


class ArrayDataset(Dataset):
    """메모리의 uint8 배열을 PIL 이미지로 바꿔 transform에 넘기는 Dataset.

    한 항목은 (이미지 텐서 (3, 32, 32), 라벨 정수) 튜플이다.
    transform이 PIL 이미지를 입력으로 받으므로 Image.fromarray로 되돌려 준다.
    같은 배열을 transform만 바꿔 여러 번 감쌀 수 있어, 증강 있는 버전과
    없는 버전이 이미지 데이터를 공유한다(메모리를 두 배로 쓰지 않는다).
    """

    def __init__(self, images, labels, transform):
        self.images = images            # (N, 32, 32, 3) uint8
        self.labels = labels            # (N,)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # images[index]: (32, 32, 3) uint8 -> PIL -> transform -> (3, 32, 32) 실수 텐서
        return self.transform(Image.fromarray(self.images[index])), int(self.labels[index])


# 채널별 평균·표준편차이므로 원소가 3개씩이다. R, G, B 순서.
# Normalize는 (픽셀 - 평균) / 표준편차로 각 채널을 평균 0, 표준편차 1 근처로 맞춘다.
# 채널마다 밝기 분포가 다른 것을 없애 학습이 더 안정적으로 진행된다.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

# base_transform: 증강 없이 텐서 변환 + 정규화만. 검증·테스트는 항상 이것을 쓴다.
# ToTensor  : PIL (32, 32) 컬러 이미지 -> (3, 32, 32) 실수 텐서, 값 범위 0~1
# Normalize : (3, 32, 32) -> (3, 32, 32)  모양은 그대로, 값 범위만 대략 -2~2로 바뀐다
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# 이미지 배열은 여기서 딱 한 번만 만든다. 아래 데이터셋들이 이 배열을 공유한다.
train_images, train_labels = load_hf_images("uoft-cs/cifar10", "train")   # (50000, 32, 32, 3)
test_images, test_labels = load_hf_images("uoft-cs/cifar10", "test")      # (10000, 32, 32, 3)
print(f"내려받은 이미지 배열: 학습 {train_images.shape}, 테스트 {test_images.shape}, "
      f"메모리 {(train_images.nbytes + test_images.nbytes) / 1024**2:.0f}MB")

full_train_base = ArrayDataset(train_images, train_labels, base_transform)
test_dataset = ArrayDataset(test_images, test_labels, base_transform)

# 학습용 5만 장을 섞은 뒤 앞 45,000장은 학습, 뒤 5,000장은 검증으로 쓴다.
# randperm(50000)은 0~49999를 섞은 (50000,) 정수 텐서. .tolist()로 파이썬 리스트로 바꾼다.
# 인덱스를 직접 만드는 이유: 뒤에서 "증강 있는 데이터셋"에도 똑같은 인덱스를 적용해야 하기 때문이다.
# 같은 이미지가 두 실험 모두에서 학습용으로 쓰여야 공정한 비교가 된다.
split_generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(len(full_train_base), generator=split_generator).tolist()
train_indices = permutation[:45000]     # 길이 45000 리스트
val_indices = permutation[45000:]       # 길이  5000 리스트
print("분할 크기(훈련/검증):", len(train_indices), len(val_indices))

# Subset(데이터셋, 인덱스 목록): 원본에서 해당 인덱스만 골라낸 부분 데이터셋
train_dataset = Subset(full_train_base, train_indices)
val_dataset = Subset(full_train_base, val_indices)

batch_size = 128
num_workers = choose_num_workers()
print(f"사용 가능한 CPU 코어에 맞춰 num_workers={num_workers}로 설정했습니다.")
# pin_memory: GPU로 옮길 때 빨라지는 메모리 영역을 쓴다(GPU가 있을 때만 의미 있음).
loader_options = {"batch_size": batch_size, "num_workers": num_workers, "pin_memory": torch.cuda.is_available()}
# DataLoader는 낱장 (3, 32, 32)을 쌓아 (B, 3, 32, 32)로 만들어 준다.
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print("데이터 수:", len(train_dataset), len(val_dataset), len(test_dataset))

# CIFAR-10의 10개 클래스. 인덱스 순서가 라벨 숫자와 일치한다(0=airplane, 1=automobile, ...).
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]


## 6주차 SmallCNN 구조 재사용
아래 `SmallCNN`은 6주차 노트북의 클래스 정의를 한 글자도 바꾸지 않고 그대로 옮긴 것입니다. 유일한 차이는 인스턴스를 만들 때 `SmallCNN(in_channels=3)`으로 채널 수만 지정한다는 점입니다. 첫 번째 합성곱 층 `nn.Conv2d(in_channels, 16, 3, padding=1)`의 입력 채널만 1에서 3으로 바뀌므로, 그 층의 파라미터 수만 늘어나고 나머지 구조는 완전히 동일합니다.


In [ ]:
class SmallCNN(nn.Module):
    """6주차와 완전히 동일한 정의. 한 글자도 바꾸지 않았다.

    달라지는 것은 인스턴스를 만들 때 in_channels=3을 주는 것뿐이고,
    그러면 첫 번째 Conv2d의 입력 채널만 1에서 3으로 바뀐다.

    아래 화살표 주석은 CIFAR-10 기준(입력 (B, 3, 32, 32))이다.
    6주차 Fashion-MNIST에서는 (B, 1, 28, 28)로 들어가 H, W가 28 -> 14 -> 7로 줄었다.
    """
    def __init__(self, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            # 여기만 흑백(1채널) 대신 컬러(3채널)를 받게 된다.
            # 가중치 모양: (16, in_channels, 3, 3). in_channels가 1이면 원소 144개, 3이면 432개.
            nn.Conv2d(in_channels, 16, 3, padding=1),   # (B, 3, 32, 32) -> (B, 16, 32, 32)
            nn.ReLU(),                                  # 모양 그대로
            nn.MaxPool2d(2),                            # (B, 16, 32, 32) -> (B, 16, 16, 16)
            nn.Conv2d(16, 32, 3, padding=1),            # (B, 16, 16, 16) -> (B, 32, 16, 16)
            nn.ReLU(),                                  # 모양 그대로
            nn.MaxPool2d(2),                            # (B, 32, 16, 16) -> (B, 32, 8, 8)
            # 입력 H, W와 무관하게 출력을 4x4로 맞춰주므로, 28x28 이미지든 32x32 이미지든
            # 뒤의 Linear 층을 고칠 필요가 없다. 그래서 구조를 그대로 재사용할 수 있다.
            nn.AdaptiveAvgPool2d((4, 4)),               # (B, 32, 8, 8) -> (B, 32, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                               # (B, 32, 4, 4) -> (B, 512)
            nn.Linear(32 * 4 * 4, 64),                  # (B, 512) -> (B, 64)
            nn.ReLU(),                                  # 모양 그대로
            nn.Linear(64, 10),                          # (B, 64) -> (B, 10)
        )

    def forward(self, x):
        # x: (B, 3, 32, 32) -> features -> (B, 32, 4, 4) -> classifier -> (B, 10)
        return self.classifier(self.features(x))

# 배치 하나를 꺼내 입력/출력 shape만 확인한다(학습은 하지 않는다).
sample_images, sample_labels = next(iter(train_loader))    # (B, 3, 32, 32), (B,)
with torch.no_grad():
    shape_logits = SmallCNN(in_channels=3).to(device)(sample_images.to(device))
print("입력 텐서 shape:", sample_images.shape)   # (128, 3, 32, 32) - 6주차는 (128, 1, 28, 28)이었다
print("출력 텐서 shape:", shape_logits.shape)     # (128, 10) - 클래스 수는 그대로 10개


## 공통 학습·평가 함수
`train_one_epoch`, `evaluate`, `fit`, `plot_history`, `count_parameters`는 6주차와 같은 역할을 합니다. 이 함수들을 세 가지 실험(기준 모델, 구조 개선 모델, 데이터 증강 모델)에서 그대로 재사용해, 모델 구조와 데이터 조건만 바뀌고 학습 절차 자체는 바뀌지 않도록 합니다.

여기에 두 가지를 추가합니다. `overfitting_gap`은 마지막 에포크의 **학습 정확도 − 검증 정확도**를 계산합니다. 이 값이 클수록 모델이 학습 데이터에만 맞춰졌다는 뜻이므로, 증강의 효과를 정확도뿐 아니라 과적합 정도로도 볼 수 있습니다. `EPOCHS`는 세 실험이 공유하는 에포크 수입니다. 에포크 수까지 똑같이 맞춰야 "구조를 바꾼 효과"와 "증강을 넣은 효과"를 다른 요인과 섞이지 않게 분리할 수 있습니다.


In [ ]:
from tqdm import tqdm

# tqdm.auto가 아니라 tqdm(글자 막대)을 쓰는 이유:
# tqdm.auto는 노트북 환경에서 ipywidgets 기반 위젯 막대를 만드는데,
# 에포크마다 새 위젯이 생기면서 메시지가 폭증해 실행이 10배 가까이 느려지는
# 경우가 있다(같은 노트북이 글자 막대 42초 -> 위젯 막대 413초로 측정됐다).
# 글자 막대도 진행률과 남은 예상 시간(ETA)을 똑같이 보여준다.

# tqdm은 반복문을 감싸 진행률과 남은 예상 시간(ETA)을 보여준다.
# tqdm.auto는 실행 환경을 보고 알아서 고른다: 노트북이면 위젯 막대, 터미널이면 글자 막대.
#
# 주의: 막대가 떠 있는 동안 print()를 쓰면 출력이 서로 섞여 막대가 깨진다.
# 그래서 에포크 요약은 print 대신 tqdm.write()로 남긴다. 내용은 동일하다.

def train_one_epoch(model, loader, criterion, optimizer, device, desc="train"):
    """학습 데이터를 한 바퀴 돌며 가중치를 갱신한다. (손실, 정확도)를 돌려준다."""
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    # leave=False: 이 배치 막대는 에포크가 끝나면 사라진다.
    for batch_images, batch_labels in tqdm(loader, desc=desc, leave=False):   # (B, 3, 32, 32), (B,)
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        optimizer.zero_grad()                       # 이전 배치의 기울기를 지운다
        logits = model(batch_images)                # 순전파: (B, 3, 32, 32) -> (B, 10)
        loss = criterion(logits, batch_labels)      # (B, 10), (B,) -> () 0차원 스칼라
        loss.backward()                             # 역전파
        optimizer.step()                            # 가중치 갱신
        loss_sum += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()   # (B, 10) -> (B,)
        total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def evaluate(model, loader, criterion, device, desc="eval"):
    """가중치를 바꾸지 않고 손실과 정확도만 잰다."""
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in tqdm(loader, desc=desc, leave=False):  # (B, 3, 32, 32), (B,)
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            logits = model(batch_images)           # (B, 10)
            loss_sum += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs, desc="전체 학습"):
    """epochs만큼 학습하며 기록을 쌓는다. 매 에포크 학습-검증 격차도 함께 출력한다.

    history의 각 값은 길이가 epochs인 파이썬 리스트다(텐서가 아니다).
    바깥쪽 tqdm 막대가 "이 실험이 언제 끝나는지"를, 안쪽 막대가
    "지금 에포크가 어디까지 갔는지"를 보여준다.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    epoch_bar = tqdm(range(1, epochs + 1), desc=desc)
    for epoch in epoch_bar:
        train_loss, train_accuracy = train_one_epoch(
            model, train_loader, criterion, optimizer, device, desc=f"[{epoch}/{epochs}] train")
        val_loss, val_accuracy = evaluate(
            model, val_loader, criterion, device, desc=f"[{epoch}/{epochs}] val")
        for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
            history[key].append(value)
        gap = train_accuracy - val_accuracy
        # set_postfix: 막대 오른쪽에 최신 수치를 붙여 둔다(막대를 새로 그리지 않는다).
        epoch_bar.set_postfix(train=f"{train_accuracy:.2f}%", val=f"{val_accuracy:.2f}%", gap=f"{gap:+.2f}%p")
        # 격차가 음수면 아직 과소적합(모델이 데이터를 덜 배운 상태),
        # 양수로 커질수록 과적합(훈련 데이터에만 맞춰지는 중)이다.
        # tqdm.write: 막대를 잠시 지웠다가 한 줄 출력하고 다시 그린다. 막대가 깨지지 않는다.
        tqdm.write(f"Epoch {epoch}/{epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%, 격차={gap:+.2f}%p")
    epoch_bar.close()
    return history

def plot_history(history, title):
    """왼쪽에 손실, 오른쪽에 정확도. 학습 곡선과 검증 곡선이 벌어지는지 보는 것이 핵심."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].legend()
    plt.show()

def count_parameters(model):
    """학습으로 값이 바뀌는 파라미터의 총 개수.

    numel()은 그 텐서의 원소 개수다. 예를 들어 Conv2d(3, 16, 3)의 가중치는
    (16, 3, 3, 3) 모양이므로 numel()은 16*3*3*3 = 432가 된다.
    """
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def overfitting_gap(history):
    """마지막 에포크의 학습 정확도 - 검증 정확도. 값이 클수록 과적합이 심하다."""
    return history["train_acc"][-1] - history["val_acc"][-1]

# 세 실험 모두 같은 에포크 수를 사용해, 성능 차이가 "구조"와 "데이터 증강"에서만 오도록 합니다.
# 22로 잡은 이유: 8에포크에서는 증강 없는 모델조차 아직 과적합에 들어가지 않아
# (학습-검증 격차가 1%p 미만) 증강이 줄여 줄 과적합 자체가 없습니다.
# 20에포크를 넘겨야 증강 없는 모델의 격차가 10%p 이상으로 벌어지면서
# 증강의 효과가 검증·테스트 정확도에서 뚜렷하게 드러납니다.
EPOCHS = 22

# 세 실험이 공유하는 손실 함수.
# (B, 10) 로짓과 (B,) 정수 라벨을 받아 0차원 스칼라 손실을 낸다.
# 내부에서 softmax를 함께 계산하므로 모델은 로짓을 그대로 내보내야 한다.
criterion = nn.CrossEntropyLoss()
print("모든 실험 공통 에포크 수:", EPOCHS)


## 기준 모델(SmallCNN) 학습
`CrossEntropyLoss`와 `Adam(lr=1e-3)`으로 `SmallCNN(in_channels=3)`을 `EPOCHS`만큼 학습해 `baseline_history`를 만듭니다. 뒤의 두 실험과 에포크 수를 똑같이 맞추므로, 이 모델과의 성능 차이는 "학습을 덜 했기 때문"이 아니라 "구조가 작기 때문"이라고 말할 수 있습니다.

### 관찰 질문
- 6주차 Fashion-MNIST에서는 짧은 학습만으로도 검증 정확도가 꽤 높게 올라갔습니다. 같은 구조의 모델이 CIFAR-10에서는 왜 검증 정확도가 낮은 수준에 머무를까요?
- 학습 정확도와 검증 정확도의 격차가 커진다면 무엇을 의심해 볼 수 있을까요?


In [ ]:
# 실험 1: 기준 모델. 6주차 구조를 채널만 바꿔 그대로 쓴다.
# seed_everything을 다시 호출하는 이유: 세 실험이 모두 "같은 무작위 초기 가중치"에서
# 출발해야 성능 차이를 구조와 증강 탓으로만 돌릴 수 있다.
seed_everything(42)

model_baseline = SmallCNN(in_channels=3).to(device)
optimizer_baseline = torch.optim.Adam(model_baseline.parameters(), lr=1e-3)
baseline_params = count_parameters(model_baseline)
print("기준 모델(SmallCNN) 파라미터 수:", baseline_params)

baseline_history = fit(model_baseline, train_loader, val_loader, criterion, optimizer_baseline, device,
                       epochs=EPOCHS, desc="실험1 기준 모델")
plot_history(baseline_history, "Baseline SmallCNN")

baseline_test_loss, baseline_test_accuracy = evaluate(model_baseline, test_loader, criterion, device, desc="기준 모델 test")
print(f"기준 모델 테스트 정확도: {baseline_test_accuracy:.2f}%")
print(f"기준 모델 과적합 격차(학습-검증): {overfitting_gap(baseline_history):+.2f}%p")


## 구조 개선 모델(ImprovedCNN) 학습
`ImprovedCNN`은 합성곱 블록을 세 단계(32, 64, 128 채널)로 늘리고, 블록마다 합성곱을 두 번 반복한 뒤 풀링합니다. `use_batchnorm`과 `dropout` 인자는 이번 실험에서 각각 `False`와 `0.0`으로 두어, 배치 정규화나 드롭아웃 없이 순수하게 "더 깊고 넓은 구조" 하나만 바꾼 효과를 관찰합니다. 학습 설정(손실 함수, 옵티마이저, 에포크 수, 데이터)은 기준 모델과 동일하게 유지합니다.

### 관찰 질문
- 에포크가 진행될수록 학습 정확도와 검증 정확도의 격차(`격차` 출력)는 어떻게 변하나요? 몇 에포크쯤부터 벌어지기 시작하나요?
- 검증 정확도가 더 이상 오르지 않는데 학습 정확도만 계속 오른다면, 모델은 무엇을 배우고 있는 걸까요?


In [ ]:
class ImprovedCNN(nn.Module):
    """SmallCNN보다 깊고 넓은 구조.

    바뀐 점 세 가지:
      1) 블록마다 합성곱을 두 번 반복한다(한 번 풀링하기 전에 특징을 두 단계로 뽑는다)
      2) 채널을 32 -> 64 -> 128로 늘려 표현력을 키운다
      3) 마지막에 AdaptiveAvgPool2d(1)로 각 채널을 숫자 하나로 요약한다

    전체 흐름(입력 (B, 3, 32, 32) 기준):
      (B,   3, 32, 32)
        -> block(3, 32)   -> (B,  32, 16, 16)
        -> block(32, 64)  -> (B,  64,  8,  8)
        -> block(64, 128) -> (B, 128,  4,  4)
        -> AdaptiveAvgPool2d(1) -> (B, 128, 1, 1)
        -> Flatten        -> (B, 128)
        -> Linear(128, 10)-> (B, 10)

    use_batchnorm과 dropout은 8주차에서 쓸 스위치다. 이번 주차에서는 둘 다 꺼서
    "더 깊고 넓은 구조" 하나만 바뀐 효과를 보려 한다.
    """
    def __init__(self, use_batchnorm=False, dropout=0.0):
        super().__init__()
        def block(in_ch, out_ch):
            """합성곱 2번 + 풀링 1번으로 이루어진 한 덩어리를 층 목록으로 만든다.

            (B, in_ch, H, W)
              -> Conv2d(in_ch, out_ch)  -> (B, out_ch, H, W)      채널만 바뀜
              -> Conv2d(out_ch, out_ch) -> (B, out_ch, H, W)      모양 그대로
              -> MaxPool2d(2)           -> (B, out_ch, H/2, W/2)  H, W만 절반
            BatchNorm2d는 모양을 바꾸지 않는다(채널별로 값만 다시 맞춘다).
            """
            layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))       # 모양 그대로
            layers += [nn.ReLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))       # 모양 그대로
            layers += [nn.ReLU(), nn.MaxPool2d(2)]          # 풀링에서만 H, W가 절반이 된다
            return layers

        self.features = nn.Sequential(
            *block(3, 32),      # (B,   3, 32, 32) -> (B,  32, 16, 16)
            *block(32, 64),     # (B,  32, 16, 16) -> (B,  64,  8,  8)
            *block(64, 128),    # (B,  64,  8,  8) -> (B, 128,  4,  4)
            # 4x4 공간 전체의 평균을 내어 채널당 값 하나로 만든다.
            # 이렇게 하면 분류기의 파라미터가 크게 줄어 과적합이 덜 생긴다.
            nn.AdaptiveAvgPool2d(1),   # (B, 128, 4, 4) -> (B, 128, 1, 1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),           # (B, 128, 1, 1) -> (B, 128)
            nn.Dropout(dropout),    # (B, 128) -> (B, 128)  dropout=0.0이면 아무 일도 하지 않는다
            nn.Linear(128, 10),     # (B, 128) -> (B, 10)
        )

    def forward(self, x):
        # x: (B, 3, 32, 32) -> features -> (B, 128, 1, 1) -> classifier -> (B, 10)
        return self.classifier(self.features(x))

# 실험 2: 구조만 바꾼다. 데이터·옵티마이저·학습률·에포크는 실험 1과 완전히 같다.
seed_everything(42)
improved_model = ImprovedCNN().to(device)
optimizer_improved = torch.optim.Adam(improved_model.parameters(), lr=1e-3)
improved_params = count_parameters(improved_model)
print("개선 모델(ImprovedCNN) 파라미터 수:", improved_params)

improved_history = fit(improved_model, train_loader, val_loader, criterion, optimizer_improved, device,
                       epochs=EPOCHS, desc="실험2 구조 개선")
plot_history(improved_history, "Improved CNN")

improved_test_loss, improved_test_accuracy = evaluate(improved_model, test_loader, criterion, device, desc="개선 모델 test")
print(f"개선 모델 테스트 정확도: {improved_test_accuracy:.2f}%")
# 이 값이 크게 나온다면 모델이 훈련 데이터를 외우기 시작했다는 뜻이다.
print(f"개선 모델 과적합 격차(학습-검증): {overfitting_gap(improved_history):+.2f}%p")


## 데이터 증강
`RandomCrop(32, padding=4)`와 `RandomHorizontalFlip()`을 훈련 데이터에만 적용합니다. 검증·테스트 데이터는 기준 전처리(`base_transform`)를 그대로 사용해, 증강이 "평가 방식"이 아니라 "훈련 방식"만 바꾼다는 점을 유지합니다. 같은 분할 인덱스(`train_indices`), 같은 `ImprovedCNN` 구조, 같은 옵티마이저 설정, 같은 에포크 수(`EPOCHS`)로 학습하므로 성능 차이는 오직 데이터 증강 여부에서만 생깁니다.

증강은 매 에포크 같은 이미지를 조금씩 다르게 잘라내고 뒤집어 보여줍니다. 모델 입장에서는 매번 처음 보는 이미지가 오는 셈이라 **훈련 데이터를 통째로 외우기가 어려워집니다**. 아래 표본 그림에서 같은 클래스의 이미지가 위치나 좌우 방향이 조금씩 어긋나 있는 것을 확인하세요.


In [ ]:
# 증강 전처리: 앞의 두 변환이 추가된 것 말고는 base_transform과 같다.
# 두 증강 모두 PIL 이미지 단계에서 적용되며, 최종 출력 모양은 (3, 32, 32)로 동일하다.
augment_transform = transforms.Compose([
    # 상하좌우에 4픽셀씩 여백을 덧댄 40x40에서 32x32를 무작위로 잘라낸다.
    # 32x32 -> (패딩) 40x40 -> (무작위 crop) 32x32. 최종 크기는 그대로다.
    # 결과적으로 물체가 매번 조금씩 다른 위치에 놓인다.
    transforms.RandomCrop(32, padding=4),
    # 50% 확률로 좌우를 뒤집는다. 크기는 바뀌지 않는다.
    # 자동차나 말은 뒤집어도 여전히 같은 클래스다.
    # (숫자나 글자였다면 뒤집으면 안 된다. 데이터에 맞는 증강을 골라야 한다.)
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),                              # -> (3, 32, 32), 값 0~1
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),        # -> (3, 32, 32), 값 대략 -2~2
])

# 같은 이미지 배열(train_images)을 transform만 바꿔 한 번 더 감싼다.
# 다시 내려받지도, 메모리를 두 배로 쓰지도 않는다. 픽셀 데이터는 하나를 공유한다.
full_train_augment = ArrayDataset(train_images, train_labels, augment_transform)
# 중요: train_indices를 그대로 재사용한다. 실험 2와 완전히 같은 45,000장이다.
augmented_train_dataset = Subset(full_train_augment, train_indices)
augmented_train_loader = DataLoader(augmented_train_dataset, shuffle=True, **loader_options)

def denormalize(tensor):
    """Normalize를 되돌린다. 정규화된 텐서는 음수를 포함해 그대로 그리면 색이 깨진다.

    입력 tensor: (3, 32, 32)   반환: (3, 32, 32)  모양은 그대로다.
    (정규화된 값 * 표준편차) + 평균 으로 원래 0~1 픽셀값을 복원하고,
    계산 오차로 범위를 벗어난 값은 clamp로 0~1 안에 가둔다.
    """
    # (3,) 튜플을 (3, 1, 1)로 만든다. 그래야 (3, 32, 32)와 곱할 때
    # 브로드캐스팅으로 "채널마다 다른 값"이 H, W 전체에 퍼진다.
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)   # (3,) -> (3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)     # (3,) -> (3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)        # (3, 32, 32) * (3, 1, 1) -> (3, 32, 32)

# 증강이 실제로 어떻게 적용되는지 눈으로 확인한다.
# 이 셀을 여러 번 실행하면 같은 클래스라도 매번 조금씩 다르게 잘리고 뒤집힌다.
augment_sample_images, augment_sample_labels = next(iter(augmented_train_loader))   # (B, 3, 32, 32), (B,)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for image, label, axis in zip(augment_sample_images[:10], augment_sample_labels[:10], axes.flat):
    # image는 낱장 (3, 32, 32).
    # permute(1,2,0): (채널, 높이, 너비) -> (높이, 너비, 채널). matplotlib은 이 순서를 요구한다.
    #                 (3, 32, 32) -> (32, 32, 3)
    axis.imshow(denormalize(image).permute(1, 2, 0))
    axis.set_title(class_names[label.item()])
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 실험 3: 실험 2와 모든 것이 같고, 훈련 로더만 증강 버전으로 바꾼다.
# 모델 구조·초기 시드·옵티마이저·학습률·에포크 수 모두 동일하므로
# 결과 차이는 오직 "훈련 데이터에 증강을 적용했는가" 하나에서만 나온다.
# 검증 로더는 val_loader(증강 없음) 그대로다. 평가 방식은 바뀌면 안 된다.
seed_everything(42)
augmented_model = ImprovedCNN().to(device)
optimizer_augmented = torch.optim.Adam(augmented_model.parameters(), lr=1e-3)
augmented_params = count_parameters(augmented_model)
# 구조가 같으므로 파라미터 수는 실험 2와 정확히 같다. 증강은 모델을 바꾸지 않는다.
print("증강 모델(ImprovedCNN + 증강) 파라미터 수:", augmented_params)

augmented_history = fit(augmented_model, augmented_train_loader, val_loader, criterion, optimizer_augmented, device,
                        epochs=EPOCHS, desc="실험3 데이터 증강")
plot_history(augmented_history, "Augmented ImprovedCNN")

augmented_test_loss, augmented_test_accuracy = evaluate(augmented_model, test_loader, criterion, device, desc="증강 모델 test")
print(f"증강 모델 테스트 정확도: {augmented_test_accuracy:.2f}%")
# 실험 2의 격차와 비교해 보자. 증강이 과적합을 얼마나 눌렀는지가 여기서 드러난다.
print(f"증강 모델 과적합 격차(학습-검증): {overfitting_gap(augmented_history):+.2f}%p")


## 세 결과 비교와 학생 활동
기준 모델, 구조 개선 모델, 데이터 증강 모델의 파라미터 수와 정확도, 그리고 **과적합 격차**를 표와 그래프로 비교합니다. 세 실험은 데이터 분할·옵티마이저·학습률·에포크 수가 모두 같으므로, 차이는 오직 "구조"와 "증강" 두 가지에서만 나옵니다.

세 번째 그래프(Overfitting Gap)를 특히 눈여겨보세요. 증강 없는 개선 모델의 격차는 에포크가 갈수록 계속 벌어지지만, 증강 모델의 격차는 낮게 유지됩니다. 검증 정확도가 정체되는데 학습 정확도만 오르는 구간이 바로 모델이 훈련 데이터를 외우기 시작한 지점입니다.

### 학생 활동
1. 세 모델의 파라미터 수를 비교하고, 파라미터가 많다고 항상 정확도가 좋아지는지 확인하세요.
2. `EPOCHS`를 8로 줄여서 다시 실행하면 증강 모델이 개선 모델을 **이기지 못합니다**. 왜 그런지, 과적합 격차 그래프를 근거로 설명해 보세요. (힌트: 8에포크 시점의 개선 모델 격차는 몇 %p인가요?)
3. `augment_transform`에서 `RandomHorizontalFlip()`만 제거하거나 `RandomCrop`만 제거해 다시 학습하고 어떤 증강이 더 중요한지 비교하세요.
4. `ImprovedCNN(use_batchnorm=True)`로 바꾸어 학습하면 오늘 결과와 어떻게 달라질지 예상하고 실제로 확인하세요.


In [ ]:
# 세 실험 결과를 한 표로 모은다.
comparison_rows = [
    ("Baseline SmallCNN", baseline_params, baseline_history, baseline_test_accuracy),
    ("Improved CNN", improved_params, improved_history, improved_test_accuracy),
    ("Augmented ImprovedCNN", augmented_params, augmented_history, augmented_test_accuracy),
]
# f-string의 :<24 는 왼쪽 정렬 24칸, :>13 은 오른쪽 정렬 13칸이라는 뜻이다.
header = f"{'모델':<24}{'파라미터 수':>13}{'최종 검증acc(%)':>15}{'최고 검증acc(%)':>15}{'테스트acc(%)':>13}{'과적합 격차(%p)':>16}"
print(header)
print("-" * 100)
for name, params, history, test_acc in comparison_rows:
    # 최종 검증acc는 마지막 에포크 값, 최고 검증acc는 전체 에포크 중 최댓값.
    # 둘이 크게 다르면 학습 후반에 검증 성능이 떨어졌다는 뜻이다.
    print(f"{name:<24}{params:>13,}{history['val_acc'][-1]:>15.2f}"
          f"{max(history['val_acc']):>15.2f}{test_acc:>13.2f}{overfitting_gap(history):>+16.2f}")

print()
# 두 효과를 분리해서 본다: 구조를 바꾼 효과와 증강을 넣은 효과
print(f"구조 개선 효과 (Improved - Baseline): 테스트 {improved_test_accuracy - baseline_test_accuracy:+.2f}%p")
print(f"데이터 증강 효과 (Augmented - Improved): 테스트 {augmented_test_accuracy - improved_test_accuracy:+.2f}%p, "
      f"과적합 격차 {overfitting_gap(augmented_history) - overfitting_gap(improved_history):+.2f}%p")

# 그래프 3개: 검증 손실, 검증 정확도, 그리고 과적합 격차의 변화
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for history, label in [
    (baseline_history, "baseline"),
    (improved_history, "improved"),
    (augmented_history, "augmented"),
]:
    epochs_axis = range(1, len(history["val_acc"]) + 1)
    axes[0].plot(epochs_axis, history["val_loss"], label=label)
    axes[1].plot(epochs_axis, history["val_acc"], label=label)
    # 에포크마다 (학습 정확도 - 검증 정확도)를 계산해 과적합이 커지는 추세를 본다.
    gaps = [t - v for t, v in zip(history["train_acc"], history["val_acc"])]
    axes[2].plot(epochs_axis, gaps, label=label)
axes[0].set_title("Validation Loss")
axes[1].set_title("Validation Accuracy")
axes[2].set_title("Overfitting Gap (train acc - val acc)")
# 0 기준선. 이 선 위로 올라가면 과적합이 시작된 것이다.
axes[2].axhline(0, color="gray", linewidth=0.8, linestyle="--")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.legend()
plt.tight_layout()
plt.show()


## 보너스: 과적합은 무엇 때문에 생길까?
위 실험에서 `ImprovedCNN`은 과적합을 보였고 `SmallCNN`은 덜 보였습니다. 여기서 "모델이 크면 과적합한다"고 외우기 쉽지만, 정확한 이야기는 조금 다릅니다. 과적합은 **데이터 양에 비해 모델의 용량이 남아돌 때** 생깁니다. 모델 크기와 데이터 양은 둘 다 한쪽 면일 뿐입니다.

이걸 확인하는 가장 빠른 방법은 모델을 그대로 두고 **데이터만 줄이는 것**입니다. 아래 실험은 앞과 완전히 같은 `ImprovedCNN`을 훈련 데이터 5,000장(전체 45,000장의 1/9)으로만 학습합니다. 증강은 쓰지 않습니다.

한 에포크가 1/9로 싸지므로 30에포크를 돌려도 앞의 실험 하나보다 훨씬 빨리 끝납니다. 그런데도 과적합 격차는 훨씬 크게 벌어집니다.

### 관찰 질문
- 같은 모델인데 데이터만 줄였을 뿐인데 왜 과적합이 빨라질까요?
- 검증 정확도의 최고값은 전체 데이터로 학습했을 때와 비교해 어떤가요? 과적합이 심한 모델이 성능도 좋을까요?
- 이 결과는 "과적합을 줄이려면 무엇을 늘려야 하는가"에 대해 무엇을 말해 주나요?


In [ ]:
# 모델·옵티마이저·학습률은 실험 2와 완전히 같다. 바꾼 것은 훈련 데이터의 "양"뿐이다.
SMALL_TRAIN_SIZE = 5000     # 전체 45,000장의 1/9
SMALL_EPOCHS = 30

# train_indices의 앞 5,000개만 쓴다. 검증에는 그대로 val_loader(5,000장)를 쓴다.
# 배치 모양은 그대로 (B, 3, 32, 32)이고, 에포크당 배치 개수만 1/9로 줄어든다.
# 증강 없는 full_train_base를 쓰는 이유: 증강 효과가 아니라 데이터 양 효과만 보려는 것이다.
small_train_loader = DataLoader(
    Subset(full_train_base, train_indices[:SMALL_TRAIN_SIZE]),
    shuffle=True, **loader_options
)

seed_everything(42)
small_data_model = ImprovedCNN().to(device)
optimizer_small = torch.optim.Adam(small_data_model.parameters(), lr=1e-3)
print(f"훈련 데이터 {SMALL_TRAIN_SIZE:,}장으로 {SMALL_EPOCHS}에포크 학습 (파라미터 수는 실험 2와 동일: {count_parameters(small_data_model):,})")

small_data_history = fit(small_data_model, small_train_loader, val_loader, criterion, optimizer_small, device,
                         epochs=SMALL_EPOCHS, desc="보너스 5,000장")

# --- 얼마나 "빨리" 과적합에 도달했는지를 비교한다 ---
# 주의: 마지막 격차의 크기만 비교하면 안 된다. 두 실험은 에포크 수도 다르고
# 한 에포크의 비용도 9배 차이 난다. 공정한 비교 기준은 "같은 연산량을 썼을 때"다.
# 그래서 격차가 처음 10%p를 넘은 시점을 찾고, 그때까지 이미지를 몇 번 통과시켰는지 센다.
GAP_THRESHOLD = 10.0

def first_epoch_reaching_gap(history, threshold):
    """학습-검증 격차가 처음으로 threshold 이상이 된 에포크 번호(1부터). 못 넘으면 None."""
    for epoch, (train_acc, val_acc) in enumerate(zip(history["train_acc"], history["val_acc"]), start=1):
        if train_acc - val_acc >= threshold:
            return epoch
    return None

full_epoch = first_epoch_reaching_gap(improved_history, GAP_THRESHOLD)
small_epoch = first_epoch_reaching_gap(small_data_history, GAP_THRESHOLD)

print()
print(f"{'실험':<32}{'훈련 이미지':>11}{'에포크':>8}{'총 통과 횟수':>14}{'최고 검증acc(%)':>16}{'최종 격차(%p)':>15}")
print("-" * 100)
print(f"{'ImprovedCNN / 45,000장 (실험 2)':<32}{len(train_indices):>11,}{EPOCHS:>8}"
      f"{EPOCHS * len(train_indices):>14,}{max(improved_history['val_acc']):>16.2f}{overfitting_gap(improved_history):>+15.2f}")
print(f"{'ImprovedCNN /  5,000장 (보너스)':<32}{SMALL_TRAIN_SIZE:>11,}{SMALL_EPOCHS:>8}"
      f"{SMALL_EPOCHS * SMALL_TRAIN_SIZE:>14,}{max(small_data_history['val_acc']):>16.2f}{overfitting_gap(small_data_history):>+15.2f}")

print(f"\n[핵심] 격차가 처음 {GAP_THRESHOLD:.0f}%p를 넘기까지 필요한 연산량")
if full_epoch and small_epoch:
    full_cost = full_epoch * len(train_indices)
    small_cost = small_epoch * SMALL_TRAIN_SIZE
    print(f"  45,000장: {full_epoch}에포크 = 이미지 {full_cost:,}번 통과")
    print(f"   5,000장: {small_epoch}에포크 = 이미지 {small_cost:,}번 통과")
    print(f"  -> 데이터를 1/9로 줄이면 같은 정도의 과적합에 약 {full_cost / small_cost:.1f}배 적은 연산으로 도달한다.")
else:
    print("  두 실험 중 한쪽이 아직 이 격차에 도달하지 못했습니다. 에포크를 늘려 다시 확인해 보세요.")

print(f"\n다만 최고 검증 정확도는 {max(improved_history['val_acc']):.2f}% -> {max(small_data_history['val_acc']):.2f}%로 크게 떨어졌다.")
print("과적합이 심한 모델이 성능도 좋은 것은 아니다. 데이터를 줄이면 과적합만 빨라지고 성능은 나빠진다.")

# 두 실험의 과적합 격차가 커지는 속도를 겹쳐 그린다.
# x축을 "에포크"가 아니라 "이미지를 몇 번 통과시켰는가"로 두어야 연산량 대비 비교가 된다.
# (matplotlib 기본 폰트에는 한글이 없으므로 범례는 영문으로 쓴다.)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for history, n_train, label in [
    (improved_history, len(train_indices), "45,000 imgs"),
    (small_data_history, SMALL_TRAIN_SIZE, "5,000 imgs"),
]:
    # 각 에포크가 끝난 시점까지의 누적 통과 횟수
    passes_axis = [epoch * n_train for epoch in range(1, len(history["val_acc"]) + 1)]
    # train_acc와 val_acc는 각각 길이가 에포크 수인 리스트. 같은 위치끼리 빼서 격차 리스트를 만든다.
    gaps = [t - v for t, v in zip(history["train_acc"], history["val_acc"])]
    axes[0].plot(passes_axis, gaps, marker="o", markersize=3, label=label)
    axes[1].plot(passes_axis, history["val_acc"], marker="o", markersize=3, label=label)
axes[0].axhline(GAP_THRESHOLD, color="red", linewidth=0.8, linestyle="--", label=f"{GAP_THRESHOLD:.0f}%p")
axes[0].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[0].set_title("Overfitting Gap vs Compute")
axes[1].set_title("Validation Accuracy vs Compute")
for axis in axes:
    axis.set_xlabel("images processed (epochs x train size)")
    axis.legend()
plt.tight_layout()
plt.show()


## 최고 성능 모델의 오분류 확인
세 모델 중 가장 성능이 좋은 `augmented_model`을 테스트셋에 적용해 오분류 이미지를 최대 10개까지 모아봅니다. 6주차와 같은 방식으로 예측과 정답을 함께 표시하되, 증강 학습에는 정규화된 텐서를 사용하므로 화면에 그리기 전에 `denormalize`로 픽셀 값을 되돌립니다.


In [ ]:
# 가장 성능이 좋은 모델이 그래도 틀리는 이미지를 모아 본다.
# 6주차와 같은 방식이지만, 두 가지가 다르다.
#   1) 평가에는 test_loader(증강 없음)를 쓴다. 증강은 학습에만 적용한다.
#   2) 이미지가 정규화되어 있으므로 그리기 전에 denormalize로 되돌려야 한다.
augmented_model.eval()

mistake_images, mistake_predictions, mistake_labels = [], [], []
with torch.no_grad():
    for batch_images, batch_labels in tqdm(test_loader, desc="오분류 수집", leave=False):   # (B, 3, 32, 32), (B,)
        batch_predictions = augmented_model(batch_images.to(device)).argmax(dim=1).cpu()
        wrong = batch_predictions != batch_labels        # 틀린 위치만 True인 마스크
        for image, prediction, label in zip(batch_images[wrong], batch_predictions[wrong], batch_labels[wrong]):
            mistake_images.append(image)
            mistake_predictions.append(prediction)
            mistake_labels.append(label)
            if len(mistake_images) == 10:
                break
        if len(mistake_images) == 10:
            break

# 테스트셋 순서대로 처음 만난 오분류 10장이다. "가장 헷갈린 10장"은 아니다.
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, image, prediction, label in zip(axes.flat, mistake_images, mistake_predictions, mistake_labels):
    axis.imshow(denormalize(image).permute(1, 2, 0))
    axis.set_title(f"Pred: {class_names[prediction.item()]}\nActual: {class_names[label.item()]}")
    axis.axis("off")
for axis in axes.flat[len(mistake_images):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 핵심 정리와 8주차 연결

### 핵심 정리
- 6주차의 `SmallCNN`을 채널 수만 바꿔 그대로 CIFAR-10에 적용하면, 흑백 의류보다 복잡한 컬러 사물 이미지 앞에서 모델 용량의 한계가 뚜렷이 드러났습니다. 세 실험의 에포크 수를 똑같이 맞췄으므로, 이 차이는 학습량이 아니라 구조에서 온 것입니다.
- 합성곱 블록을 깊고 넓게 만든 `ImprovedCNN`은 같은 데이터·같은 학습 설정에서도 기준 모델보다 높은 정확도를 보여, 구조 자체의 표현력이 성능에 미치는 영향을 확인했습니다. 다만 학습이 진행될수록 학습 정확도만 계속 오르고 검증 정확도는 정체되는 **과적합**이 나타났습니다.
- `RandomCrop`과 `RandomHorizontalFlip`으로 학습 데이터만 증강한 `augmented_model`은 같은 `ImprovedCNN` 구조에서 과적합 격차를 크게 줄이고, 그 결과 검증·테스트 정확도도 더 높였습니다.
- **증강의 효과는 과적합이 시작된 뒤에야 나타납니다.** 학습 초반에는 증강이 오히려 학습 속도를 늦춰 정확도가 낮게 보입니다. 아직 외울 만큼 학습하지도 않은 모델에게 "외우지 말라"고 하는 규제는 손해일 뿐이기 때문입니다. 규제 기법의 효과를 판단할 때는 반드시 충분히 학습한 뒤에 비교해야 한다는 점을 기억하세요.
- 오분류 이미지들은 여전히 모양이나 배경이 비슷한 클래스끼리 헷갈리는 경향을 보여줍니다.

### 8주차 연결 질문
오늘 세 모델은 모두 같은 학습률(`1e-3`)과 같은 옵티마이저(Adam)를 사용했습니다. 8주차에서는 학습률과 옵티마이저(SGD·Adam) 자체를 짧게 비교하고, `ImprovedCNN`에 배치 정규화와 드롭아웃을 더했을 때 학습 안정성과 과적합 방지에 어떤 차이가 생기는지, 그리고 Early Stopping이 왜 마지막 에포크가 아니라 가장 좋았던 가중치를 선택하는지 살펴봅니다. 오늘 관찰한 과적합 격차 그래프를 떠올리며, 배치 정규화·드롭아웃이 그 격차를 어떻게 줄여줄지 미리 예상해 보세요. 또 오늘 "증강 없는 모델은 검증 정확도가 정체된 뒤에도 계속 학습했다"는 점이, 8주차의 Early Stopping이 필요한 이유와 어떻게 이어지는지도 생각해 보세요.
